In [1]:
import sys, random
import numpy as np
import warnings
import fastf1
import pandas as pd
import requests

RANDOM_SEED = 414  # Course constant. Do not change.
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
warnings.filterwarnings('ignore', category=FutureWarning)

# Environment check
print(f'Python  : {sys.version.split()[0]}')
print(f'NumPy   : {np.__version__}')
print(f'Seed    : {RANDOM_SEED}')

Python  : 3.13.11
NumPy   : 2.4.2
Seed    : 414


In [2]:
session = fastf1.get_session(2024, 'Abu Dhabi', 'Q')

print(session)

session.load(laps=True, telemetry=True, weather=True, messages=True)


print('\n----DATA----\nNumber of laps in the results:',session.results.shape)

columns = [x for x in session.results.columns]

print('Coolumns in dataset:', columns)

display(session.laps.head())

print('Shape:',session.laps.shape)

req         WARNING 	DEFAULT CACHE ENABLED! (45.48 MB) /home/vicente/.cache/fastf1
core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info


2024 Season Round 24: Abu Dhabi Grand Prix - Qualifying


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['4', '81', '55', '27', '1', '10', '63', '14', '77', '11', '22', '30', '18', '16', '20', '23', '24', '44', '43', '61']



----DATA----
Number of laps in the results: (20, 22)
Coolumns in dataset: ['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'Position', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'Time', 'Status', 'Points', 'Laps']


,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,...,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate
0,0 days 00:22:41.925000,NOR,4,NaT,1.0,1.0,0 days 00:20:03.650000,NaT,NaT,0 days 00:00:49.350000,...,True,McLaren,0 days 00:20:03.650000,2024-12-07 14:05:03.653,1,NaN,False,,False,False
1,0 days 00:24:05.607000,NOR,4,0 days 00:01:23.682000,2.0,1.0,NaT,NaT,0 days 00:00:17.129000,0 days 00:00:36.301000,...,True,McLaren,0 days 00:22:41.925000,2024-12-07 14:07:41.928,1,NaN,False,,False,True
2,0 days 00:25:57.551000,NOR,4,0 days 00:01:51.944000,3.0,1.0,NaT,0 days 00:25:56.445000,0 days 00:00:21.676000,0 days 00:00:47.997000,...,True,McLaren,0 days 00:24:05.607000,2024-12-07 14:09:05.610,1,NaN,False,,False,False
3,0 days 00:31:52.979000,NOR,4,NaT,4.0,2.0,0 days 00:29:10.623000,NaT,NaT,0 days 00:00:46.011000,...,True,McLaren,0 days 00:25:57.551000,2024-12-07 14:10:57.554,1,NaN,False,,False,False
4,0 days 00:33:25.776000,NOR,4,0 days 00:01:32.797000,5.0,2.0,NaT,0 days 00:33:24.624000,0 days 00:00:17.017000,0 days 00:00:36.024000,...,True,McLaren,0 days 00:31:52.979000,2024-12-07 14:16:52.982,1,NaN,False,,False,False


Shape: (261, 31)


In [8]:
url = "https://api.jolpi.ca/ergast/f1/2024/driverstandings.json"

response = requests.get(url)

status_code = response.status_code
print(f"HTTP Status Code: {status_code}")

if status_code == 200:
    data = response.json()
    
    standings_list = data['MRData']['StandingsTable']['StandingsLists'][0]['DriverStandings']
    
    print(f"Number of records returned: {len(standings_list)}")
    print("-" * 30)
    
    df = pd.DataFrame(standings_list)
    
    print("Shape:", df.shape)


    print("Top 3 Driver Entries:")
    for entry in standings_list[:3]:
        points = entry['points']
        driver = entry['Driver']
        name = f"{driver['givenName']} {driver['familyName']}"
        nationality = driver['nationality']
        
        print(f"• {name} ({nationality}): {points} points")
else:
    print("Error al consultar la API")

HTTP Status Code: 200
Number of records returned: 24
------------------------------
Shape: (24, 6)
Top 3 Driver Entries:
• Max Verstappen (Dutch): 437 points
• Lando Norris (British): 374 points
• Charles Leclerc (Monegasque): 356 points


1. I chose 'fastf1.get_session(2024, 'Abu Dhabi', 'Q')' because it was used in the development of the other jupyter lab

2. Datasets shape's:
    FastF1: (261, 31)
    Jilpica: (24, 6)

3. One interesting aspect of the dataset structure is the difference in the shapes of the two datasets. The FastF1 dataset has a shape of (261, 31). In contrast, the Jolpica (Ergast) dataset has a shape of (24, 6), as it summarizes higher-level information about drivers in the standings.

